# Beer Predictor — Data Preparation

Loads the curated raw catalog (`data/raw/curated_beers.csv`), cleans it, engineers a 7-component **taste vector** for each beer (all 0–10), and saves the result as a parquet file at `data/processed/beers.parquet` for the recommender and dashboard to consume.

Taste-vector components: `bitterness`, `sweetness`, `maltiness`, `hoppiness`, `body`, `sourness`, `abv_normalized`.

Style baselines come from the BJCP 2021 style guidelines (publicly available factual descriptors); IBU and ABV refine the bitterness/hoppiness/strength axes.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.style_attributes import STYLE_ATTRS, style_lookup, normalize_style

RAW = ROOT / 'data' / 'raw' / 'curated_beers.csv'
OUT = ROOT / 'data' / 'processed' / 'beers.parquet'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Raw file exists:', RAW.exists())

Project root: C:\Users\dvhee\projects-outside-onedrive\beer-predictor
Raw file exists: True


## 1. Load raw data

In [2]:
raw = pd.read_csv(RAW)
print(raw.shape)
raw.head()

(588, 8)


,beer_id,name,brewery,style,country,abv,ibu,srm
0,cervecera-jerome__rubia,Rubia,Cervecera Jerome,American Lager,Argentina,6.0,NaN,NaN
1,cervecera-jerome__cerveza-negra,Cerveza Negra,Cervecera Jerome,American Stout,Argentina,6.5,NaN,NaN
2,cervecera-jerome__cerveza-roja,Cerveza Roja,Cervecera Jerome,Märzen,Argentina,6.0,NaN,NaN
3,cervecera-jerome__cerveza-diablo,Cerveza Diablo,Cervecera Jerome,American Lager,Argentina,7.5,NaN,NaN
4,brouwerij-nacional-balashi__balashi,Balashi,Brouwerij Nacional Balashi,German Pilsner,Aruba,5.0,NaN,NaN


## 2. Clean

- Drop full-row duplicates and exact name+brewery duplicates.
- Normalize style names (no-op on the curated set, useful for future external data).
- Coerce numeric ABV/IBU/SRM; clip ABV to [0, 20] and IBU to [0, 120] as plausible bounds.
- For any missing IBU we fall back to a style baseline (mean IBU of beers sharing the style).

In [3]:
df = raw.copy()
df = df.drop_duplicates()
df = df.drop_duplicates(subset=['name', 'brewery'])
df['style'] = df['style'].astype(str).str.strip().map(normalize_style)
for c in ('abv', 'ibu', 'srm'):
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['abv'] = df['abv'].clip(lower=0, upper=20)
df['ibu'] = df['ibu'].clip(lower=0, upper=120)

# Backfill missing IBU with the style mean, then with global mean.
df['ibu'] = df.groupby('style')['ibu'].transform(lambda s: s.fillna(s.mean()))
df['ibu'] = df['ibu'].fillna(df['ibu'].mean())
df['srm'] = df['srm'].fillna(df.groupby('style')['srm'].transform('mean')).fillna(df['srm'].mean())
df = df.dropna(subset=['abv'])
print('After cleaning:', df.shape)
df.head()

After cleaning: (588, 8)


,beer_id,name,brewery,style,country,abv,ibu,srm
0,cervecera-jerome__rubia,Rubia,Cervecera Jerome,American Lager,Argentina,6.0,14.000000,4.000000
1,cervecera-jerome__cerveza-negra,Cerveza Negra,Cervecera Jerome,American Stout,Argentina,6.5,40.000000,38.000000
2,cervecera-jerome__cerveza-roja,Cerveza Roja,Cervecera Jerome,Märzen,Argentina,6.0,26.177305,10.576923
3,cervecera-jerome__cerveza-diablo,Cerveza Diablo,Cervecera Jerome,American Lager,Argentina,7.5,14.000000,4.000000
4,brouwerij-nacional-balashi__balashi,Balashi,Brouwerij Nacional Balashi,German Pilsner,Aruba,5.0,26.000000,3.500000


## 3. Engineer the taste vector

All seven components are placed on the same 0–10 scale.

- **bitterness**: `clip(IBU / 10, 0, 10)` (IBU 0 → 0, IBU 100 → 10).
- **sweetness, maltiness, body, sourness**: pulled directly from the BJCP style baseline.
- **hoppiness**: blend of style baseline and IBU-derived bitterness — `0.5 * style + 0.5 * (IBU/10)`.
- **abv_normalized**: `clip(ABV / 12 * 10, 0, 10)` (12% ABV → 10).

In [4]:
def attrs_for_row(row: pd.Series) -> pd.Series:
    sweet, malt, hop_base, body, sour = style_lookup(row['style'])
    ibu = float(row['ibu']) if pd.notna(row['ibu']) else np.nan
    abv = float(row['abv']) if pd.notna(row['abv']) else 0.0

    if np.isnan(ibu):
        bitterness = float(hop_base)
    else:
        bitterness = float(np.clip(ibu / 10.0, 0, 10))

    hoppiness = float(np.clip(0.5 * hop_base + 0.5 * bitterness, 0, 10))
    abv_norm = float(np.clip(abv / 12.0 * 10.0, 0, 10))

    return pd.Series({
        'bitterness': round(bitterness, 2),
        'sweetness': float(sweet),
        'maltiness': float(malt),
        'hoppiness': round(hoppiness, 2),
        'body': float(body),
        'sourness': float(sour),
        'abv_normalized': round(abv_norm, 2),
    })

taste = df.apply(attrs_for_row, axis=1)
df = pd.concat([df.reset_index(drop=True), taste.reset_index(drop=True)], axis=1)
TASTE_COLS = ['bitterness', 'sweetness', 'maltiness', 'hoppiness', 'body', 'sourness', 'abv_normalized']
df[TASTE_COLS].describe()

,bitterness,sweetness,maltiness,hoppiness,body,sourness,abv_normalized
count,588.000000,588.000000,588.000000,588.000000,588.000000,588.000000,588.000000
mean,2.618912,4.608844,5.809524,3.436293,5.641156,0.418367,5.182755
std,0.882397,1.536625,1.837832,1.180878,1.802555,1.163823,1.640494
min,0.800000,2.000000,2.000000,0.900000,2.000000,0.000000,0.420000
25%,2.600000,3.000000,4.000000,2.810000,4.000000,0.000000,4.170000
50%,2.620000,5.000000,6.000000,3.310000,6.000000,0.000000,4.580000
75%,2.620000,6.000000,7.000000,4.250000,7.000000,1.000000,6.010000
max,6.000000,8.000000,9.000000,7.000000,9.000000,8.000000,10.000000


## 4. Sanity checks

Every taste-vector value must lie within `[0, 10]`, and there should be no NaNs in the taste columns.

In [5]:
assert df[TASTE_COLS].isna().sum().sum() == 0, 'NaNs in taste columns'
assert (df[TASTE_COLS] >= 0).all().all() and (df[TASTE_COLS] <= 10).all().all(), 'taste vector out of range'
assert df['abv'].between(0, 20).all(), 'ABV out of plausible range'
print('Sanity checks passed for', len(df), 'beers across', df['style'].nunique(), 'styles')

Sanity checks passed for 588 beers across 50 styles


## 5. Save processed catalog

In [6]:
OUT_COLS = ['beer_id', 'name', 'brewery', 'style', 'country', 'abv', 'ibu', 'srm'] + TASTE_COLS
df_out = df[OUT_COLS].copy()
df_out.to_parquet(OUT, index=False)
print('Wrote', OUT, '— shape', df_out.shape)
df_out.head(8)

Wrote

 C:\Users\dvhee\projects-outside-onedrive\beer-predictor\data\processed\beers.parquet — shape (588, 15)


,beer_id,name,brewery,style,country,abv,ibu,srm,bitterness,sweetness,maltiness,hoppiness,body,sourness,abv_normalized
0,cervecera-jerome__rubia,Rubia,Cervecera Jerome,American Lager,Argentina,6.0,14.000000,4.000000,1.40,3.0,3.0,1.70,3.0,0.0,5.00
1,cervecera-jerome__cerveza-negra,Cerveza Negra,Cervecera Jerome,American Stout,Argentina,6.5,40.000000,38.000000,4.00,5.0,7.0,5.00,7.0,0.0,5.42
2,cervecera-jerome__cerveza-roja,Cerveza Roja,Cervecera Jerome,Märzen,Argentina,6.0,26.177305,10.576923,2.62,5.0,7.0,2.81,6.0,0.0,5.00
3,cervecera-jerome__cerveza-diablo,Cerveza Diablo,Cervecera Jerome,American Lager,Argentina,7.5,14.000000,4.000000,1.40,3.0,3.0,1.70,3.0,0.0,6.25
4,brouwerij-nacional-balashi__balashi,Balashi,Brouwerij Nacional Balashi,German Pilsner,Aruba,5.0,26.000000,3.500000,2.60,2.0,4.0,4.30,3.0,0.0,4.17
5,coopers-brewery__coopers-premium-lager,Coopers Premium Lager,Coopers Brewery,American Lager,Australia,5.0,14.000000,4.000000,1.40,3.0,3.0,1.70,3.0,0.0,4.17
6,coopers-brewery__coopers-premium-light,Coopers Premium Light,Coopers Brewery,American Light Lager,Australia,2.9,8.000000,3.000000,0.80,2.0,2.0,0.90,2.0,0.0,2.42
7,coopers-brewery__coopers-best-extra-stout,Coopers Best Extra Stout,Coopers Brewery,American Stout,Australia,6.3,40.000000,38.000000,4.00,5.0,7.0,5.00,7.0,0.0,5.25


## 6. Quick eyeball: highest-bitterness vs highest-sweetness

In [7]:
print('Top bitterness:')
print(df_out.sort_values('bitterness', ascending=False).head(5)[['name', 'style', 'bitterness']].to_string(index=False))
print('\nTop sweetness:')
print(df_out.sort_values('sweetness', ascending=False).head(5)[['name', 'style', 'sweetness']].to_string(index=False))
print('\nTop sourness:')
print(df_out.sort_values('sourness', ascending=False).head(5)[['name', 'style', 'sourness']].to_string(index=False))

Top bitterness:
                   name        style  bitterness
  King's Blockhouse IPA American IPA        6.00
                    IPA American IPA        5.75
                    IPA American IPA        5.75
          Trafalgar IPA American IPA        5.75
Anniversary IPA Ahtanum American IPA        5.75

Top sweetness:
                name       style  sweetness
  Mackeson XXX Stout Sweet Stout        8.0
        Rochefort 10   Quadrupel        8.0
Left Hand Milk Stout Sweet Stout        8.0
                Quad   Quadrupel        8.0
           Blasphemy   Quadrupel        8.0

Top sourness:
                     name        style  sourness
Mort Subite Gueuze Lambic Fruit Lambic       8.0
Cassis Reserve Lambic '06 Fruit Lambic       8.0
Wild Blue Blueberry Lager Fruit Lambic       8.0
              Gueuze 1882 Fruit Lambic       8.0
            Arizona Peach Fruit Lambic       8.0
